## Librerias

In [158]:
import pandas as pd

## Carga de datos

In [159]:
df = pd.read_csv('../../data/data_cleaning/clean_data_09-03-2026.csv',parse_dates=['insert_date'])
df

C:\Users\Usuario\AppData\Local\Temp\ipykernel_14144\2939003354.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv('../../data/data_cleaning/clean_data_09-03-2026.csv',parse_dates=['insert_date'])


,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,NaN,Private room,2,2.0,1.0,...,100.0,100.0,100.0,100.0,100.0,FALSO,75.0,spain,malaga,2018-07-31
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,...,90.0,100.0,100.0,80.0,90.0,FALSO,52.0,spain,madrid,2020-01-10
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,2.0,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,142.0,spain,sevilla,2019-07-29
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,1.0,...,90.0,100.0,100.0,100.0,90.0,VERDADERO,306.0,spain,barcelona,2020-01-10
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,NaN,Private room,5,1.0,2.0,...,100.0,100.0,100.0,100.0,100.0,FALSO,39.0,spain,girona,2019-02-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6996,27241318,ES MOLI D'EN SION - Villa with private pool in...,Enjoy the peace of the countryside in this bea...,80839530,Sa Pobla,NaN,Entire home/apt,10,4.0,5.0,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,7.0,spain,mallorca,2020-04-23
6997,27244243,101.108_New building apartment with two double...,Apartment in Cadaqu�s center. 1rst �floor. Ele...,151496825,Cadaqu�s,NaN,Entire home/apt,4,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,2018-08-30
6998,27244794,101.38_Apartment with one doble bedroom and te...,101.38.- Apartment placed Sa T�rtora � Sant An...,151496825,Cadaqu�s,NaN,Entire home/apt,2,1.0,1.0,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,6.0,spain,girona,2019-12-31
6999,27245117,MATILLA - Fant�stico apartamento con garaje,Apartamento espacioso a 7 minutos del centro d...,137859766,Cadaqu�s,NaN,Entire home/apt,6,2.0,3.0,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,2018-07-31


## Marketing

## Cliente

## Operaciones
- Oferta total disponible
- Porcentaje de oferta disponible
- Disponibles en el próximo mes
- Completos/reservados
- Ratio de ocupación mensual
- Ratio de ocupación anual

In [160]:
# Creación del df para operaciones
df_operaciones = df.copy()

availability_cols = [
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365"
]

# apartamentos totalmente ocupados
df_operaciones["fully_booked"] = (df_operaciones[availability_cols] == 0).all(axis=1)

# ratios de ocupación
df_operaciones["occ_month"] = (30 - df_operaciones["availability_30"]) / 30
df_operaciones["occ_2month"] = (60 - df_operaciones["availability_60"]) / 60
df_operaciones["occ_3month"] = (90 - df_operaciones["availability_90"]) / 90
df_operaciones["occ_year"] = (365 - df_operaciones["availability_365"]) / 365

occupancy_cols = [
    "occ_month",
    "occ_2month",
    "occ_3month",
    "occ_year"
]
df_operaciones['has_availability'].value_counts()

has_availability
VERDADERO    6451
Name: count, dtype: int64

In [161]:
# Oferta total disponible
total_anuncios = df_operaciones["apartment_id"].nunique()                       # 6615 anuncios
apartamentos_disponibles = (                                                    # 6087 apartamentos disponibles
    df_operaciones["apartment_id"].count() - 
    (df_operaciones["has_availability"]==False).sum())                          
available_30 = (df_operaciones["availability_30"] > 0).sum()                            # 4661 son apartamentos disponibles en el próximo mes
ratio_oferta_disponible = round((available_30 / apartamentos_disponibles) * 100, 2)     # Hay un 70.9% de apartamentos disponibles en el próximo mes
fully_booked_but_active = df_operaciones[
    (df_operaciones["fully_booked"]) & 
    (df_operaciones["has_availability"])
].shape[0]                                                                      # 813 apartamentos estan ocupados (anuncio está activo)


In [162]:
pd.crosstab(
    df_operaciones["fully_booked"],
    df_operaciones["has_availability"]
)

has_availability,VERDADERO
fully_booked,
False,5608
True,843


In [163]:
# Oferta disponible por tipo de habitación
oferta_tipo = (
    df_operaciones[df_operaciones["availability_30"] > 0]
    .groupby("room_type")["apartment_id"]
    .nunique()
    .reset_index(name="alojamientos_disponibles")
    .sort_values(by="alojamientos_disponibles", ascending=False)   
)
oferta_tipo

,room_type,alojamientos_disponibles
0,Entire home/apt,3555
2,Private room,1188
3,Shared room,30
1,Hotel room,24


In [164]:
# Oferta disponible por ciudad
oferta_ciudad = (
    df_operaciones[df_operaciones["availability_30"] > 0]
    .groupby("city")["apartment_id"]
    .nunique()
    .reset_index(name="alojamientos_disponibles")
    .sort_values(by="alojamientos_disponibles", ascending=False)
)
oferta_ciudad

,city,alojamientos_disponibles
0,barcelona,1444
2,madrid,923
4,mallorca,815
1,girona,749
6,sevilla,292
3,malaga,258
7,valencia,221
5,menorca,94


In [165]:
# Disponobilidad media general
disponibilidad_media = df_operaciones[availability_cols].mean().round(0)
disponibilidad_media

availability_30      12.0
availability_60      27.0
availability_90      44.0
availability_365    188.0
dtype: float64

In [166]:
# Ratio de ocupación estimada (30,60,90,365)
ratio_ocupacion = (df_operaciones[occupancy_cols].mean() * 100).round(2)
ratio_ocupacion

occ_month     59.08
occ_2month    54.30
occ_3month    50.77
occ_year      48.46
dtype: float64

In [167]:
ocupacion_ciudad = (
    df_operaciones
    .groupby("city")[occupancy_cols]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)
ocupacion_ciudad

,city,occ_month,occ_2month,occ_3month,occ_year
0,barcelona,63.11,57.30,52.70,49.68
1,girona,51.29,47.76,46.20,46.47
2,madrid,65.42,60.05,56.01,55.52
3,malaga,59.30,52.33,47.33,44.36
4,mallorca,55.01,51.82,49.42,41.84
5,menorca,50.09,48.52,46.86,45.07
6,sevilla,53.87,49.13,45.01,44.78
7,valencia,54.93,50.61,46.99,49.25


In [168]:
kpi_operaciones = {
    "anuncios_totales": int(total_anuncios),
    "apartamentos_disponibles": int(apartamentos_disponibles),
    "available_supply_%": float(ratio_oferta_disponible),
    "available_next_30_days": int(available_30),
    "fully_booked": int(fully_booked_but_active),
    "occupancy_30_days_%": float(ratio_ocupacion["occ_month"]),
    "occupancy_year_%": float(ratio_ocupacion["occ_year"])
}
kpi_operaciones_df = pd.DataFrame(
    list(kpi_operaciones.items()),
    columns=["KPI", "Valor"]
)

kpi_operaciones_df

,KPI,Valor
0,anuncios_totales,6733.00
1,apartamentos_disponibles,7001.00
2,available_supply_%,70.90
3,available_next_30_days,4964.00
4,fully_booked,843.00
5,occupancy_30_days_%,59.08
6,occupancy_year_%,48.46
